# YOLO-Fastest 人物検出モデル学習 (darknet版) - Ethos-U55 NPU対応

darknetフレームワークでYOLO-Fastest V1を学習し、TFLite INT8に変換する。

## Phase 2 (Issue #133) の主な変更点

mAP@0.50 35.13% (#131) -> **50%以上** を目標とした Phase 2 改善:

| 項目 | #131 | **Phase 2 (#133)** |
|---|---|---|
| データセットクリーニング | なし | **Step 2.5 で品質フィルタ + 重複除去** |
| max_batches | 100,000 | **200,000 (約200エポック)** |
| learning rate policy | step decay | **SGDR (cosine annealing)** |
| burn_in | 2,000 | **4,000** |
| mosaic augmentation | 無効 | **有効 (darknet 内蔵)** |
| mixup augmentation | 無効 | **有効 (darknet 内蔵)** |
| cutmix augmentation | 無効 | **有効 (darknet 内蔵)** |
| jitter | 0.3 | **0.5** |
| saturation / exposure | 1.5 | **2.0** |
| hue | 0.1 | **0.15** |
| resume marker | .issue131_started | **.issue133_started** |

## Phase 2 の狙い
1. **2A データセット品質向上**: ぼけ/暗所/破損ラベル/重複を除去し、学習の上限を引き上げる
2. **2B augmentation 強化**: mosaic/mixup/cutmix + 色空間拡張でデータの多様性を拡大
3. **2C 学習延長**: cosine annealing で後半の最適化を滑らかにし、100k 以降の収束不足を解消

## 改善版 (Issue #131) の主な変更点

| 項目 | 旧版 (#128) | **改善版 (#131)** |
|---|---|---|
| 事前学習重み | なし (スクラッチ) | **COCO事前学習済みbackboneで転移学習** |
| max_batches | 10,000 (約10エポック) | **100,000 (約100エポック)** |
| 学習率 | 固定step decay | **Warmup + step decay (burn_in=2000)** |
| データ拡張 | デフォルト | **angle=15, ignore_thresh=0.5** |
| アンカー | 顔検出placeholder | **K-meansで人物データセットに最適化** |
| MCUパラメータ | 手動確認 | **自動出力セル追加** |

## 精度低下の原因分析 (#128)
1. スクラッチ学習 (事前学習済み重みなし) -> 転移学習で解決
2. 学習不足 (50,000 iteration / 約50エポック) -> 100,000 iterationに増加
3. アンカーが顔検出リファレンスのplaceholder値のまま -> K-means再計算

## モデル仕様
- ベース: dog-qiuqiu/Yolo-Fastest (YOLO-Fastest V1)
- 入力: 192x192x3 (RGB, INT8)
- 出力: 2ブランチ (6x6 stride-32 + 12x12 stride-16), 各3アンカー x 6値 (x,y,w,h,obj,cls)
- クラス: 1 (person)
- 検出方式: アンカーベース (YOLOv3スタイル)

## 前提条件
- Google Colab (GPU: T4以上)
- Google Driveに `fall_detection_dataset.zip` をアップロード済み

## ワークフロー
1. GPU確認・環境構築 (darknetコンパイル)
2. データセット準備 (darknet形式)
2.5. **Phase 2 データセットクリーニング (新規)**
3. YOLO-Fastest cfg作成 (Phase 2 パラメータ)
4. アンカーK-means計算
5. 事前学習済み重み取得 (COCO backbone)
6. モデル学習 (200,000 iterations, cosine annealing)
7. 精度評価
8. darknet weights -> TFLite INT8 変換
9. モデル詳細確認 + MCUパラメータ出力
10. Vela互換性確認
11. 結果保存・ダウンロード

---
## Step 1: GPU確認・環境構築

**重要:** メニューの「ランタイム > ランタイムのタイプを変更」で **GPU (T4)** を選択してください。

darknetをGPU対応でコンパイルします。

In [ ]:
# GPU 確認
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
    print('=== GPU が利用可能です ===')
else:
    print('WARNING: GPU が検出されません。ランタイムを GPU に変更してください。')

In [ ]:
# Yolo-Fastest リポジトリをクローンしてdarknetをコンパイル
import os

DARKNET_DIR = '/content/Yolo-Fastest'

if not os.path.isdir(DARKNET_DIR):
    !git clone https://github.com/dog-qiuqiu/Yolo-Fastest.git {DARKNET_DIR}
else:
    print(f'{DARKNET_DIR} は既に存在します')

# darknetサブディレクトリの確認
# Yolo-Fastestリポジトリはdarknetのフォークを含む
darknet_makefile = os.path.join(DARKNET_DIR, 'Makefile')
if os.path.exists(darknet_makefile):
    print('Makefile が見つかりました')
else:
    # リポジトリ構造確認
    !ls -la {DARKNET_DIR}/
    print('\nWARNING: Makefileが見つかりません。リポジトリ構造を確認してください。')

In [ ]:
%%bash
# darknetをGPU対応でコンパイル
cd /content/Yolo-Fastest

# MakefileをGPU対応に修正
sed -i 's/GPU=0/GPU=1/' Makefile
sed -i 's/CUDNN=0/CUDNN=1/' Makefile
# OpenCVは学習に不要。Colab環境ではopencv-devが未インストールのため無効化
# デフォルトがOPENCV=1の場合もあるので、明示的に0に設定
sed -i 's/OPENCV=1/OPENCV=0/' Makefile

# コンパイル
make clean 2>/dev/null || true
make -j$(nproc)

# 確認
if [ -f ./darknet ]; then
    echo ''
    echo '=== darknet コンパイル成功 ==='
    ls -la ./darknet
else
    echo 'ERROR: darknet コンパイル失敗'
fi


---
## Step 2: データセット準備

### 事前準備 (ローカルPCで実行)

```bash
cd mimamori-sense/dataset/merged
zip -r fall_detection_dataset.zip images/ labels/
```

作成した `fall_detection_dataset.zip` を Google Drive のマイドライブ直下にアップロードしてください。

### darknet形式について

darknetは以下のファイルを必要とします:
- `obj.data`: クラス数、パス等の設定
- `obj.names`: クラス名一覧
- `train.txt`: 学習画像の絶対パス一覧
- `valid.txt`: 検証画像の絶対パス一覧
- 各画像と同名の `.txt` ラベルファイル (YOLO形式: class_id x_center y_center w h)

In [ ]:
# Google Drive マウント
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import glob

DARKNET_DIR = '/content/Yolo-Fastest'
DATASET_DIR = '/content/dataset'
DATASET_ZIP = '/content/drive/MyDrive/fall_detection_dataset.zip'
DATA_DIR = os.path.join(DARKNET_DIR, 'data', 'person')
BACKUP_DIR = os.path.join(DARKNET_DIR, 'backup')
GDRIVE_BACKUP = '/content/drive/MyDrive/yolo_fastest_darknet_person'

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(BACKUP_DIR, exist_ok=True)
os.makedirs(GDRIVE_BACKUP, exist_ok=True)

# データセット展開
if not os.path.isdir(os.path.join(DATASET_DIR, 'images')):
    if os.path.isfile(DATASET_ZIP):
        print('データセット展開中...')
        !mkdir -p {DATASET_DIR} && unzip -q {DATASET_ZIP} -d {DATASET_DIR}
        print('展開完了')
    else:
        print(f'ERROR: {DATASET_ZIP} が見つかりません')
        print('Google Driveに fall_detection_dataset.zip をアップロードしてください')
else:
    print('データセットは展開済みです')

# 検証
for split in ['train', 'val', 'test']:
    img_dir = os.path.join(DATASET_DIR, 'images', split)
    lbl_dir = os.path.join(DATASET_DIR, 'labels', split)
    if os.path.isdir(img_dir):
        img_count = len([f for f in os.listdir(img_dir)
                         if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
        lbl_count = len([f for f in os.listdir(lbl_dir)
                         if f.endswith('.txt')]) if os.path.isdir(lbl_dir) else 0
        print(f'  {split}: images={img_count}, labels={lbl_count}')

In [ ]:
import os
import glob

DARKNET_DIR = '/content/Yolo-Fastest'
DATASET_DIR = '/content/dataset'
DATA_DIR = os.path.join(DARKNET_DIR, 'data', 'person')

print('=== darknet形式ファイル生成 ===')

# ----- obj.names -----
names_path = os.path.join(DATA_DIR, 'obj.names')
with open(names_path, 'w') as f:
    f.write('person\n')
print(f'作成: {names_path}')

# ----- darknet用のラベル配置 -----
# darknetは画像と同じディレクトリに.txtラベルを置く規約
# 既存データセットはimages/とlabels/が分離しているのでシンボリックリンクを作成
for split in ['train', 'val', 'test']:
    img_dir = os.path.join(DATASET_DIR, 'images', split)
    lbl_dir = os.path.join(DATASET_DIR, 'labels', split)
    if not os.path.isdir(img_dir) or not os.path.isdir(lbl_dir):
        continue
    linked = 0
    for lbl_file in os.listdir(lbl_dir):
        if not lbl_file.endswith('.txt'):
            continue
        src = os.path.join(lbl_dir, lbl_file)
        dst = os.path.join(img_dir, lbl_file)
        if not os.path.exists(dst):
            os.symlink(src, dst)
            linked += 1
    print(f'  {split}: {linked} ラベルファイルをリンク済み')

# ----- train.txt / valid.txt -----
for split, filename in [('train', 'train.txt'), ('val', 'valid.txt')]:
    img_dir = os.path.join(DATASET_DIR, 'images', split)
    if not os.path.isdir(img_dir):
        print(f'WARNING: {img_dir} が見つかりません')
        continue
    images = sorted(glob.glob(os.path.join(img_dir, '*.jpg')) +
                    glob.glob(os.path.join(img_dir, '*.png')) +
                    glob.glob(os.path.join(img_dir, '*.jpeg')))
    list_path = os.path.join(DATA_DIR, filename)
    with open(list_path, 'w') as f:
        for img_path in images:
            f.write(img_path + '\n')
    print(f'作成: {list_path} ({len(images)} 画像)')

# ----- obj.data -----
data_path = os.path.join(DATA_DIR, 'obj.data')
with open(data_path, 'w') as f:
    f.write(f'classes = 1\n')
    f.write(f'train = {os.path.join(DATA_DIR, "train.txt")}\n')
    f.write(f'valid = {os.path.join(DATA_DIR, "valid.txt")}\n')
    f.write(f'names = {names_path}\n')
    f.write(f'backup = {os.path.join(DARKNET_DIR, "backup")}\n')
print(f'作成: {data_path}')

# ----- 確認 -----
print('\n=== obj.data 内容 ===')
with open(data_path) as f:
    print(f.read())

# ラベルの整合性確認 (先頭5ファイル)
print('=== ラベル整合性確認 (先頭5件) ===')
train_list = os.path.join(DATA_DIR, 'train.txt')
with open(train_list) as f:
    lines = f.readlines()[:5]
for line in lines:
    img_path = line.strip()
    # darknetは画像パスの拡張子を.txtに置換してラベルを探す
    lbl_path = os.path.splitext(img_path)[0] + '.txt'
    img_ok = os.path.exists(img_path)
    lbl_ok = os.path.exists(lbl_path)
    print(f'  img={img_ok} lbl={lbl_ok} {os.path.basename(img_path)}')

---
## Step 2.5: Phase 2 データセットクリーニング (Issue #133)

**Phase 2 で新規追加**: データセットの品質上限が mAP を制約している可能性があるため、学習前に以下をクリーニングする。

### クリーニング内容
1. **ぼけ画像の除去** - Laplacian variance が閾値未満
2. **輝度異常画像の除去** - 暗すぎ / 白飛び
3. **ラベル異常の除去** - 座標範囲外、サイズ0、画像/ラベル対応ミス
4. **重複画像の除去** - perceptual hash (ahash) + Hamming 距離

### 実行ポリシー
- まず dry-run で統計を確認
- 想定内であれば `--apply` で実際に退避 (物理削除ではなく `_removed_phase2/` に移動)
- 退避ファイルは切り戻し可能

### 期待効果
- ラベルノイズ除去で Precision 向上 (目標 60%+)
- 重複除去で overfit を抑制
- `mAP@0.50` の上振れ余地を 3-5 pt 追加

In [ ]:
# === Phase 2 (#133) データセットクリーニング ===
# dataset_cleaning.py を Google Drive 経由でアップロードして実行する

import os
import subprocess
import sys

DATASET_DIR = '/content/dataset'
CLEANING_SCRIPT = '/content/dataset_cleaning.py'

# リポジトリからスクリプトを取得 (Google Drive にコピー済みの想定)
# もし存在しなければ Drive からコピー
if not os.path.exists(CLEANING_SCRIPT):
    drive_script = '/content/drive/MyDrive/yolo_fastest_darknet_person/dataset_cleaning.py'
    if os.path.exists(drive_script):
        import shutil
        shutil.copy2(drive_script, CLEANING_SCRIPT)
        print(f'クリーニングスクリプトを Drive からコピー: {CLEANING_SCRIPT}')
    else:
        print('ERROR: dataset_cleaning.py が見つかりません')
        print('  mimamori-sense/dataset/scripts/dataset_cleaning.py を')
        print(f'  {drive_script} にアップロードしてください')

if os.path.exists(CLEANING_SCRIPT):
    # --- Step 2.5.1: dry-run (統計のみ) ---
    print('=' * 60)
    print('[Phase 2] Dry-run: クリーニング候補の統計を確認')
    print('=' * 60)
    cmd_dry = [
        sys.executable, CLEANING_SCRIPT,
        '--dataset', DATASET_DIR,
        '--blur-threshold', '50',
        '--dark-threshold', '25',
        '--bright-threshold', '235',
        '--dup-hash-size', '8',
        '--dup-hamming-threshold', '4',
        '--report', '/content/phase2_cleaning_report.json',
    ]
    subprocess.run(cmd_dry, check=False)

    # --- Step 2.5.2: 実適用 (退避) ---
    # 結果を確認して問題なければ以下を有効化する
    APPLY_CLEANING = True  # False にすると dry-run のみ
    if APPLY_CLEANING:
        print()
        print('=' * 60)
        print('[Phase 2] Apply: 検出ファイルを _removed_phase2/ に退避')
        print('=' * 60)
        cmd_apply = cmd_dry + ['--apply']
        subprocess.run(cmd_apply, check=False)

        # train.txt / valid.txt を再生成 (退避後の画像リスト反映)
        print()
        print('--- train.txt / valid.txt を再生成 ---')
        import glob
        DATA_DIR = '/content/Yolo-Fastest/data/person'
        for split, filename in [('train', 'train.txt'), ('val', 'valid.txt')]:
            img_dir = os.path.join(DATASET_DIR, 'images', split)
            if not os.path.isdir(img_dir):
                continue
            images = sorted(
                glob.glob(os.path.join(img_dir, '*.jpg')) +
                glob.glob(os.path.join(img_dir, '*.png')) +
                glob.glob(os.path.join(img_dir, '*.jpeg'))
            )
            list_path = os.path.join(DATA_DIR, filename)
            with open(list_path, 'w') as f:
                for img_path in images:
                    f.write(img_path + '\n')
            print(f'  {filename}: {len(images)} 画像 (クリーニング後)')

    # レポート JSON を Google Drive にバックアップ
    report_json = '/content/phase2_cleaning_report.json'
    if os.path.exists(report_json):
        import shutil
        gdrive_report = '/content/drive/MyDrive/yolo_fastest_darknet_person/phase2_cleaning_report.json'
        shutil.copy2(report_json, gdrive_report)
        print(f'\nレポートを Drive に保存: {gdrive_report}')
else:
    print('\nSKIP: dataset_cleaning.py が使えないため Phase 2 クリーニングをスキップします')
    print('      (#131 と同じ品質のまま学習します)')


---
## Step 3: YOLO-Fastest cfg作成 (Phase 2 #133)

YOLO-Fastest V1 のcfgを人物検出用にカスタマイズします。

### Phase 2 (#133) の主な変更点
- **max_batches: 200,000** (旧: 100,000) - さらに学習延長
- **policy: sgdr** (cosine annealing) - step decay より滑らかな収束
- **burn_in: 4,000** (旧: 2,000) - 大きな learning rate への立ち上げ
- **mosaic=1 / mixup=1 / cutmix=1** - darknet 内蔵の強力な augmentation
- **jitter=0.5** (旧: 0.3) - ランダムクロップ強化
- **saturation=2.0 / exposure=2.0** (旧: 1.5) - 色空間拡張強化
- **hue=0.15** (旧: 0.1)
- **ignore_thresh=0.5** (#131 から維持) - 学習信号確保
- **angle=15** (#131 から維持) - 回転拡張
- classes: 1 (person)
- 入力: 192x192x3 (RGB)
- filters (YOLO前): 18 = (1+5)*3

In [ ]:
import os

DARKNET_DIR = '/content/Yolo-Fastest'
CFG_DIR = os.path.join(DARKNET_DIR, 'cfg')
os.makedirs(CFG_DIR, exist_ok=True)

# Yolo-Fastestリポジトリ内のcfgファイルを確認
print('=== リポジトリ内のcfgファイル ===')
for root, dirs, files in os.walk(DARKNET_DIR):
    for f in files:
        if f.endswith('.cfg'):
            print(f'  {os.path.relpath(os.path.join(root, f), DARKNET_DIR)}')

In [ ]:
import os

DARKNET_DIR = '/content/Yolo-Fastest'
CFG_DIR = os.path.join(DARKNET_DIR, 'cfg')
os.makedirs(CFG_DIR, exist_ok=True)

# YOLO-Fastest V1 person検出用cfg (192x192x3 RGB)
# Phase 2 (#133): max_batches 200k, cosine annealing, mosaic/mixup/cutmix, 色空間拡張強化
# アンカーはデフォルト値。Step 4でデータセットに合わせて再計算する

CFG_PATH = os.path.join(CFG_DIR, 'yolo-fastest-person-192.cfg')

cfg_content = """[net]
batch=64
subdivisions=16
width=192
height=192
channels=3
momentum=0.9
decay=0.0005
angle=15
saturation=2.0
exposure=2.0
hue=.15

learning_rate=0.001
burn_in=4000
max_batches=200000
policy=sgdr
sgdr_cycle=1000
sgdr_mult=2

# Phase 2 (#133) darknet 内蔵 augmentation
mosaic=1
mixup=1
cutmix=1

# 0 - Backbone: Shufflenet v2 0.5x
[convolutional]
batch_normalize=1
filters=16
size=3
stride=2
pad=1
activation=leaky

[maxpool]
size=2
stride=2

# 2 - Channel split
[convolutional]
batch_normalize=1
filters=8
size=1
stride=1
pad=1
activation=leaky

[convolutional]
batch_normalize=1
filters=8
size=3
stride=1
pad=1
groups=8
activation=linear

[convolutional]
batch_normalize=1
filters=8
size=1
stride=1
pad=1
activation=leaky

# 5 - Route and shuffle
[route]
layers=-1,-4

[convolutional]
batch_normalize=1
filters=16
size=1
stride=1
pad=1
activation=leaky

# 7 - Stride 2 block
[convolutional]
batch_normalize=1
filters=16
size=3
stride=2
pad=1
groups=16
activation=linear

[convolutional]
batch_normalize=1
filters=16
size=1
stride=1
pad=1
activation=leaky

[route]
layers=-3

[convolutional]
batch_normalize=1
filters=16
size=1
stride=1
pad=1
activation=leaky

[convolutional]
batch_normalize=1
filters=16
size=3
stride=2
pad=1
groups=16
activation=linear

[convolutional]
batch_normalize=1
filters=16
size=1
stride=1
pad=1
activation=leaky

# 14 - Route
[route]
layers=-1,-5

[convolutional]
batch_normalize=1
filters=32
size=1
stride=1
pad=1
activation=leaky

# 16 - Repeat block
[convolutional]
batch_normalize=1
filters=16
size=1
stride=1
pad=1
activation=leaky

[convolutional]
batch_normalize=1
filters=16
size=3
stride=1
pad=1
groups=16
activation=linear

[convolutional]
batch_normalize=1
filters=16
size=1
stride=1
pad=1
activation=leaky

[route]
layers=-1,-4

[convolutional]
batch_normalize=1
filters=32
size=1
stride=1
pad=1
activation=leaky

# 22 - Stride 2 block (to 12x12)
[convolutional]
batch_normalize=1
filters=32
size=3
stride=2
pad=1
groups=32
activation=linear

[convolutional]
batch_normalize=1
filters=24
size=1
stride=1
pad=1
activation=leaky

[route]
layers=-3

[convolutional]
batch_normalize=1
filters=24
size=1
stride=1
pad=1
activation=leaky

[convolutional]
batch_normalize=1
filters=24
size=3
stride=2
pad=1
groups=24
activation=linear

[convolutional]
batch_normalize=1
filters=24
size=1
stride=1
pad=1
activation=leaky

[route]
layers=-1,-5

[convolutional]
batch_normalize=1
filters=48
size=1
stride=1
pad=1
activation=leaky

# 30 - Repeat blocks
[convolutional]
batch_normalize=1
filters=24
size=1
stride=1
pad=1
activation=leaky

[convolutional]
batch_normalize=1
filters=24
size=3
stride=1
pad=1
groups=24
activation=linear

[convolutional]
batch_normalize=1
filters=24
size=1
stride=1
pad=1
activation=leaky

[route]
layers=-1,-4

[convolutional]
batch_normalize=1
filters=48
size=1
stride=1
pad=1
activation=leaky

# 35 - Stride 2 block (to 6x6)
[convolutional]
batch_normalize=1
filters=48
size=3
stride=2
pad=1
groups=48
activation=linear

[convolutional]
batch_normalize=1
filters=48
size=1
stride=1
pad=1
activation=leaky

[route]
layers=-3

[convolutional]
batch_normalize=1
filters=48
size=1
stride=1
pad=1
activation=leaky

[convolutional]
batch_normalize=1
filters=48
size=3
stride=2
pad=1
groups=48
activation=linear

[convolutional]
batch_normalize=1
filters=48
size=1
stride=1
pad=1
activation=leaky

[route]
layers=-1,-5

[convolutional]
batch_normalize=1
filters=96
size=1
stride=1
pad=1
activation=leaky

# 43 - Repeat block
[convolutional]
batch_normalize=1
filters=48
size=1
stride=1
pad=1
activation=leaky

[convolutional]
batch_normalize=1
filters=48
size=3
stride=1
pad=1
groups=48
activation=linear

[convolutional]
batch_normalize=1
filters=48
size=1
stride=1
pad=1
activation=leaky

[route]
layers=-1,-4

[convolutional]
batch_normalize=1
filters=96
size=1
stride=1
pad=1
activation=leaky

######## YOLO Head ########

# 48 - Detection head branch 0 (6x6, stride 32)
[convolutional]
batch_normalize=1
filters=48
size=1
stride=1
pad=1
activation=leaky

[convolutional]
size=1
stride=1
pad=1
filters=18
activation=linear

[yolo]
mask=3,4,5
anchors=10,14, 23,27, 37,58, 81,82, 135,169, 344,319
classes=1
num=6
jitter=.5
ignore_thresh=.5
truth_thresh=1
random=0
scale_x_y=1.05

# 51 - Upsample and route for branch 1 (12x12, stride 16)
[route]
layers=-3

[upsample]
stride=2

# Route to 12x12 feature map from backbone
[route]
layers=-1,29

# 54 - Detection head branch 1 (12x12, stride 16)
[convolutional]
batch_normalize=1
filters=48
size=1
stride=1
pad=1
activation=leaky

[convolutional]
size=1
stride=1
pad=1
filters=18
activation=linear

[yolo]
mask=0,1,2
anchors=10,14, 23,27, 37,58, 81,82, 135,169, 344,319
classes=1
num=6
jitter=.5
ignore_thresh=.5
truth_thresh=1
random=0
scale_x_y=1.05
"""

with open(CFG_PATH, 'w') as f:
    f.write(cfg_content)

print(f'cfg作成: {CFG_PATH}')
print()
print('=== Phase 2 (#133) の主な変更点 ===')
print(f'  入力: 192x192x3 (RGB)')
print(f'  クラス数: 1 (person)')
print(f'  出力 filters: 18 = (1+5)*3')
print(f'  max_batches: 200,000 (旧 #131: 100,000)')
print(f'  policy: sgdr (cosine annealing, sgdr_cycle=1000, sgdr_mult=2)')
print(f'  burn_in: 4,000 (旧 #131: 2,000)')
print(f'  mosaic=1, mixup=1, cutmix=1 (darknet 内蔵 augmentation)')
print(f'  jitter: 0.5 (旧 #131: 0.3)')
print(f'  saturation=2.0, exposure=2.0 (旧 #131: 1.5)')
print(f'  hue: 0.15 (旧 #131: 0.1)')
print(f'  angle: 15 (#131 から維持)')
print(f'  ignore_thresh: 0.5 (#131 から維持)')
print(f'  batch: 64, subdivisions: 16')
print()
print('NOTE: アンカーはデフォルト値です。Step 4でデータセットに合わせて再計算します。')
print()
print('NOTE: この Phase 2 cfg は手書きテンプレートです。')
print('リポジトリ内の yolo-fastest-1.1.cfg がある場合はそちらをベースに修正することを推奨します。')
print('次のセルでリポジトリ内のcfgをベースにした版も作成します。')


In [ ]:
import os
import re

DARKNET_DIR = '/content/Yolo-Fastest'
CFG_DIR = os.path.join(DARKNET_DIR, 'cfg')

# リポジトリ内の基準cfgをベースに人物検出用に修正する版
# yolo-fastest-1.1.cfg または yolo-fastest-1.1_body.cfg を探す
base_cfg = None
for candidate in [
    os.path.join(DARKNET_DIR, 'ModelZoo', 'yolo-fastest-1.1_body', 'yolo-fastest-1.1_body.cfg'),
    os.path.join(DARKNET_DIR, 'ModelZoo', 'yolo-fastest-1.1_coco', 'yolo-fastest-1.1.cfg'),
    os.path.join(DARKNET_DIR, 'cfg', 'yolo-fastest-1.1.cfg'),
    os.path.join(DARKNET_DIR, 'yolo-fastest-1.1.cfg'),
]:
    if os.path.exists(candidate):
        base_cfg = candidate
        break

if base_cfg:
    print(f'ベースcfg: {base_cfg}')
    with open(base_cfg) as f:
        content = f.read()

    # 入力サイズを192x192に変更
    content = re.sub(r'width=\d+', 'width=192', content)
    content = re.sub(r'height=\d+', 'height=192', content)

    # channels=3 (RGB)
    content = re.sub(r'channels=\d+', 'channels=3', content)

    # classes=1
    content = re.sub(r'classes=\d+', 'classes=1', content)

    # --- Phase 2 (#133): 学習パラメータ最適化 ---
    # max_batches: 200,000 (約200エポック)
    content = re.sub(r'max_batches\s*=\s*\d+', 'max_batches=200000', content)

    # policy: SGDR (cosine annealing)
    # 既存の policy=steps を置換し、steps/scales を削除
    content = re.sub(r'policy\s*=\s*\w+', 'policy=sgdr', content)
    # steps / scales 行を削除 (sgdr では不要)
    content = re.sub(r'^steps\s*=.*$', '', content, flags=re.MULTILINE)
    content = re.sub(r'^scales\s*=.*$', '', content, flags=re.MULTILINE)
    # sgdr_cycle / sgdr_mult を learning_rate 直後に追加
    if 'sgdr_cycle' not in content:
        content = re.sub(
            r'(learning_rate\s*=\s*[\d.]+)',
            r'\1\nburn_in=4000\nsgdr_cycle=1000\nsgdr_mult=2',
            content,
            count=1,
        )
    # burn_in を 4,000 に調整
    content = re.sub(r'burn_in\s*=\s*\d+', 'burn_in=4000', content)

    # 色空間 augmentation 強化
    content = re.sub(r'saturation\s*=\s*[\d.]+', 'saturation=2.0', content)
    content = re.sub(r'exposure\s*=\s*[\d.]+', 'exposure=2.0', content)
    content = re.sub(r'^hue\s*=\s*[\d.]+', 'hue=.15', content, flags=re.MULTILINE)

    # 回転augmentation維持 (#131 からの引き継ぎ)
    content = re.sub(r'angle\s*=\s*\d+', 'angle=15', content)

    # darknet 内蔵 augmentation 有効化
    # [net] セクション内に mosaic/mixup/cutmix を追加 (存在しなければ)
    if 'mosaic=' not in content:
        content = re.sub(
            r'(\[net\][^\[]*?)(\n\[)',
            r'\1\nmosaic=1\nmixup=1\ncutmix=1\2',
            content,
            count=1,
            flags=re.DOTALL,
        )

    # ignore_thresh を下げて学習信号を増やす (#131 から維持)
    content = re.sub(r'ignore_thresh\s*=\s*[\d.]+', 'ignore_thresh=.5', content)

    # jitter 強化
    content = re.sub(r'jitter\s*=\s*[\d.]+', 'jitter=.5', content)

    # [yolo]レイヤー前の[convolutional]のfiltersを修正
    # filters = (classes + 5) * num_anchors_per_branch = (1+5)*3 = 18
    lines = content.split('\n')
    new_lines = []
    i = 0
    while i < len(lines):
        if i < len(lines) - 1 and lines[i].strip() == '[yolo]':
            for j in range(len(new_lines) - 1, -1, -1):
                if 'filters=' in new_lines[j] and not new_lines[j].strip().startswith('#'):
                    new_lines[j] = 'filters=18'
                    break
                if new_lines[j].strip().startswith('[') and new_lines[j].strip() != '[convolutional]':
                    break
        new_lines.append(lines[i])
        i += 1

    content = '\n'.join(new_lines)

    # 保存
    CFG_PATH = os.path.join(CFG_DIR, 'yolo-fastest-person-192.cfg')
    with open(CFG_PATH, 'w') as f:
        f.write(content)
    print(f'修正版cfg保存: {CFG_PATH}')
    print()
    print('=== Phase 2 (#133) 適用した修正 ===')
    print('  width/height: 192')
    print('  channels: 3')
    print('  classes: 1')
    print('  max_batches: 200,000')
    print('  policy: sgdr (cosine annealing)')
    print('  sgdr_cycle: 1000, sgdr_mult: 2')
    print('  burn_in: 4,000')
    print('  mosaic=1, mixup=1, cutmix=1')
    print('  jitter: 0.5')
    print('  saturation: 2.0, exposure: 2.0, hue: 0.15')
    print('  angle: 15')
    print('  ignore_thresh: 0.5')
    print('  YOLO前のfilters: 18')
else:
    print('WARNING: リポジトリ内にベースcfgが見つかりませんでした')
    print('前のセルで作成した手書きテンプレートを使用してください')


---
## Step 4: アンカー計算

データセットのバウンディングボックス分布に最適化したアンカーを計算します。

計算後、cfgファイルのanchors行を更新してください。

In [ ]:
%%bash
cd /content/Yolo-Fastest

# darknet calc_anchors で最適アンカーを計算
# 6アンカー (2ブランチ x 3アンカー)
./darknet detector calc_anchors \
    data/person/obj.data \
    -num_of_clusters 6 \
    -width 192 -height 192 \
    -show 2>&1 || echo 'NOTE: calc_anchorsが失敗した場合、以下のPythonセルで計算します'

In [ ]:
# Pythonでアンカー計算 (darknet calc_anchorsのフォールバック)
import numpy as np
import os
import glob

DATASET_DIR = '/content/dataset'
IMG_SIZE = 192
NUM_CLUSTERS = 6

# 全ラベルからバウンディングボックスのw,hを収集
boxes = []
for split in ['train']:
    lbl_dir = os.path.join(DATASET_DIR, 'labels', split)
    if not os.path.isdir(lbl_dir):
        continue
    for lbl_file in sorted(os.listdir(lbl_dir)):
        if not lbl_file.endswith('.txt'):
            continue
        with open(os.path.join(lbl_dir, lbl_file)) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    w = float(parts[3]) * IMG_SIZE
                    h = float(parts[4]) * IMG_SIZE
                    if w > 0 and h > 0:
                        boxes.append([w, h])

boxes = np.array(boxes)
print(f'総ボックス数: {len(boxes)}')
print(f'W 統計: min={boxes[:,0].min():.1f}, max={boxes[:,0].max():.1f}, mean={boxes[:,0].mean():.1f}')
print(f'H 統計: min={boxes[:,1].min():.1f}, max={boxes[:,1].max():.1f}, mean={boxes[:,1].mean():.1f}')

# K-means clustering (IoUベース)
def iou_distance(box, clusters):
    """1 - IoU distance"""
    w = np.minimum(box[0], clusters[:, 0])
    h = np.minimum(box[1], clusters[:, 1])
    intersection = w * h
    box_area = box[0] * box[1]
    cluster_area = clusters[:, 0] * clusters[:, 1]
    union = box_area + cluster_area - intersection
    return 1.0 - intersection / union

def kmeans_anchors(boxes, k, max_iter=300):
    n = len(boxes)
    # 初期クラスタ中心をランダムに選択
    np.random.seed(42)
    indices = np.random.choice(n, k, replace=False)
    clusters = boxes[indices].copy()

    for _ in range(max_iter):
        distances = np.array([iou_distance(box, clusters) for box in boxes])
        assignments = np.argmin(distances, axis=1)

        new_clusters = np.zeros_like(clusters)
        for i in range(k):
            mask = assignments == i
            if mask.sum() > 0:
                new_clusters[i] = boxes[mask].mean(axis=0)
            else:
                new_clusters[i] = clusters[i]

        if np.allclose(clusters, new_clusters):
            break
        clusters = new_clusters

    # 面積でソート
    areas = clusters[:, 0] * clusters[:, 1]
    order = np.argsort(areas)
    return clusters[order]

anchors = kmeans_anchors(boxes, NUM_CLUSTERS)

print(f'\n=== 計算されたアンカー ({NUM_CLUSTERS}個) ===')
anchor_str_parts = []
for i, (w, h) in enumerate(anchors):
    print(f'  [{i}] w={w:.0f}, h={h:.0f} (area={w*h:.0f})')
    anchor_str_parts.append(f'{w:.0f},{h:.0f}')

anchor_str = ', '.join(anchor_str_parts)
print(f'\ndarknet cfg形式:')
print(f'anchors = {anchor_str}')
print(f'\nBranch 1 (12x12, stride 16): mask=0,1,2 -> {anchor_str_parts[0]}, {anchor_str_parts[1]}, {anchor_str_parts[2]}')
print(f'Branch 0 (6x6,  stride 32): mask=3,4,5 -> {anchor_str_parts[3]}, {anchor_str_parts[4]}, {anchor_str_parts[5]}')
print()
print('*** cfgファイルのanchors行を上記の値で更新してください ***')

In [ ]:
# cfgファイルのアンカーを更新
import re

CFG_PATH = '/content/Yolo-Fastest/cfg/yolo-fastest-person-192.cfg'

# anchor_str は前のセルで計算済み
try:
    with open(CFG_PATH) as f:
        cfg = f.read()

    # anchors行を置換
    cfg_updated = re.sub(r'anchors\s*=\s*[\d, .]+', f'anchors={anchor_str}', cfg)

    with open(CFG_PATH, 'w') as f:
        f.write(cfg_updated)

    print(f'cfg更新完了: {CFG_PATH}')
    print(f'anchors = {anchor_str}')
except NameError:
    print('ERROR: anchor_str が定義されていません。前のセルを先に実行してください。')

---
## Step 5: 事前学習済み重み取得 (改善版 #131)

COCO事前学習済みのYOLO-Fastest backboneを取得して転移学習のベースとします。

### 重みの取得方法 (優先度順)
1. **COCO学習済みモデルのフル重みで転移学習**
   - darknetはcfg不一致の層を自動的にスキップして読み込む
   - backbone + neck 部分が転移される
2. **backboneの抽出** (`darknet partial`)
   - COCO学習済み重みからbackbone部分のみ抽出して転移学習
3. **スクラッチ学習** (フォールバック)
   - 事前学習済み重みが取得できない場合

### 重要
- dog-qiuqiu/Yolo-Fastest のModelZooにCOCO学習済み重みあり
- Google Driveからのダウンロードが必要な場合がある (URLが変わる可能性)

In [ ]:
import os
import glob

DARKNET_DIR = '/content/Yolo-Fastest'

# リポジトリ内の重みファイルを探す
print('=== リポジトリ内の重みファイル ===')
weights_files = glob.glob(os.path.join(DARKNET_DIR, '**/*.weights'), recursive=True)
for wf in weights_files:
    size_kb = os.path.getsize(wf) / 1024
    print(f'  {os.path.relpath(wf, DARKNET_DIR)}: {size_kb:.1f} KB')

if not weights_files:
    print('  重みファイルが見つかりません。ダウンロードを試みます。')

In [ ]:
%%bash
cd /content/Yolo-Fastest

# === 改善版 (#131): COCO事前学習済み重みの取得 ===
# 優先度: (1) ModelZoo内の重み -> (2) ダウンロード -> (3) スクラッチ

echo "=== COCO事前学習済み重み取得 ==="

PRETRAINED=""
CFG_FOR_PARTIAL=""

# パターン1: ModelZoo内の重みを探す
for wf in $(find ModelZoo -name '*.weights' 2>/dev/null); do
    echo "  発見: $wf ($(du -k "$wf" | cut -f1) KB)"
    # COCO版 (yolo-fastest-1.1) を優先
    case "$wf" in
        *coco*|*yolo-fastest-1.1.weights)
            PRETRAINED="$wf"
            # 対応するcfgも探す
            cfg_dir=$(dirname "$wf")
            for cf in "$cfg_dir"/*.cfg; do
                if [ -f "$cf" ]; then
                    CFG_FOR_PARTIAL="$cf"
                fi
            done
            ;;
    esac
    if [ -z "$PRETRAINED" ]; then
        PRETRAINED="$wf"
        cfg_dir=$(dirname "$wf")
        for cf in "$cfg_dir"/*.cfg; do
            if [ -f "$cf" ]; then
                CFG_FOR_PARTIAL="$cf"
            fi
        done
    fi
done

# パターン2: ダウンロード試行
if [ -z "$PRETRAINED" ]; then
    echo ""
    echo "ModelZoo内に重みなし。ダウンロードを試みます..."
    # dog-qiuqiu の releases や Google Drive からダウンロード
    # NOTE: URLは変更される可能性があるため、失敗時はスクラッチに切り替え
    echo "NOTE: 自動ダウンロードが失敗した場合は手動で重みを配置してください"
    echo "  参照: https://github.com/dog-qiuqiu/Yolo-Fastest"
fi

echo ""

if [ -n "$PRETRAINED" ]; then
    echo "=== 事前学習済み重み: $PRETRAINED ==="
    ls -la "$PRETRAINED"

    # --- 方法A: backbone抽出 (darknet partial) ---
    echo ""
    echo "--- backbone重みの抽出 (darknet partial) ---"

    # partial には元のcfgが必要
    if [ -n "$CFG_FOR_PARTIAL" ]; then
        echo "元のcfg: $CFG_FOR_PARTIAL"
        # backbone最終層までのレイヤー番号を指定
        # YOLO-Fastest V1: backbone部分は約47層
        ./darknet partial "$CFG_FOR_PARTIAL" "$PRETRAINED" \
            yolo-fastest-backbone.conv 47 2>&1 && \
            echo "backbone抽出成功: yolo-fastest-backbone.conv" || \
            echo "WARNING: partial抽出失敗"
    fi

    # --- 方法B: フル重みを転移学習用に保持 ---
    # darknetは不一致層を自動スキップするため、フル重みでもtrain可能
    echo ""
    echo "--- フル重みも転移学習に使用可能 ---"
    echo "darknetはcfg不一致の層を自動的にスキップして読み込みます"
    echo "classes数が違ってもbackbone + neck部分は転移されます"
    cp "$PRETRAINED" yolo-fastest-coco-pretrained.weights
    echo "コピー: yolo-fastest-coco-pretrained.weights"
else
    echo "=== スクラッチ学習を実行します ==="
    echo "事前学習済み重みが見つからないため、ランダム初期化から学習します"
    echo "精度改善のために事前学習済み重みの使用を強く推奨します"
fi

---
## Step 6: モデル学習 (Phase 2 #133)

darknet detector train で学習を実行します。

### Phase 2 (#133) の改善ポイント
- **max_batches: 200,000** (旧: 100,000) - 約200エポックの十分な学習
- **burn_in: 4,000** - 大きな learning rate への安全な立ち上げ
- **policy=sgdr (cosine annealing)** - 後半の滑らかな収束で 100k 以降の停滞を解消
- **mosaic=1 / mixup=1 / cutmix=1** - darknet 内蔵の強力な augmentation
- **色空間拡張強化**: saturation/exposure=2.0, hue=0.15, jitter=0.5
- **転移学習**: #131 の best weights (_best.weights) または COCO backbone から開始

### 学習時間目安
- T4 GPU: 約7-10時間 (200,000 iterations, mosaic/mixup 有効で若干遅くなる)
- A100 GPU: 約2-4時間

### 接続切れからの再開
学習結果は Google Drive に直接コピーされます。
接続が切れた場合:
1. ランタイムを再起動
2. Step 1-5 を再実行 (データセット準備まで)
3. Step 6 のセルを再実行 (自動的に `_last.weights` から再開)

### .issue133_started マーカー
Phase 2 での新規学習か再開かを判別するために `.issue133_started` マーカーを使用します。
- 初回実行: マーカーを作成し、#131 の重みまたは COCO backbone から新規開始
- 再実行: マーカーが存在すれば `_last.weights` から再開
- **#131 の重みを Phase 2 の転移学習元にしたい場合**: `yolo-fastest-person-192_best.weights` を
  `/content/drive/MyDrive/yolo_fastest_darknet_person/backup/` にコピーしておき、
  マーカーを作らず (削除して) Step 6 を実行 -> 新規開始モードで #131 best から学習が再開される

In [ ]:
import os
import shutil
import glob

DARKNET_DIR = '/content/Yolo-Fastest'
BACKUP_DIR = os.path.join(DARKNET_DIR, 'backup')
GDRIVE_BACKUP = '/content/drive/MyDrive/yolo_fastest_darknet_person'
CFG_PATH = os.path.join(DARKNET_DIR, 'cfg', 'yolo-fastest-person-192.cfg')
DATA_PATH = os.path.join(DARKNET_DIR, 'data', 'person', 'obj.data')

# Issue #133 (Phase 2) 学習開始マーカー
# このファイルが存在する場合、#133 Phase 2 の学習が開始済み → _last.weightsからの再開を許可
# 存在しない場合、#131 までの重みは無視して Phase 2 事前学習済み重みから新規開始
ISSUE133_MARKER = os.path.join(GDRIVE_BACKUP, '.issue133_started')

os.makedirs(BACKUP_DIR, exist_ok=True)
os.makedirs(GDRIVE_BACKUP, exist_ok=True)
os.makedirs(os.path.join(GDRIVE_BACKUP, 'backup'), exist_ok=True)

# backupディレクトリをGoogle Driveへのシンボリックリンクに置換
# darknetが保存する重みが自動的にGoogle Driveに書き込まれる (Colab切断対策)
gdrive_backup_dir = os.path.join(GDRIVE_BACKUP, 'backup')
if os.path.islink(BACKUP_DIR):
    print(f'backupディレクトリは既にGoogle Driveへのシンボリックリンクです')
elif os.path.isdir(BACKUP_DIR):
    import glob as _g
    for w in _g.glob(os.path.join(BACKUP_DIR, '*.weights')):
        dst = os.path.join(gdrive_backup_dir, os.path.basename(w))
        if not os.path.exists(dst):
            shutil.copy2(w, dst)
            print(f'  コピー: {os.path.basename(w)} -> Google Drive')
    shutil.rmtree(BACKUP_DIR)
    os.symlink(gdrive_backup_dir, BACKUP_DIR)
    print(f'backup -> Google Drive シンボリックリンク作成済み')
else:
    os.symlink(gdrive_backup_dir, BACKUP_DIR)
    print(f'backup -> Google Drive シンボリックリンク作成済み')

# --- 重み選択ロジック ---
local_last = os.path.join(BACKUP_DIR, 'yolo-fastest-person-192_last.weights')
backbone_weights = os.path.join(DARKNET_DIR, 'yolo-fastest-backbone.conv')
coco_weights = os.path.join(DARKNET_DIR, 'yolo-fastest-coco-pretrained.weights')
pretrained_weights = ''

is_issue133_started = os.path.exists(ISSUE133_MARKER)

if is_issue133_started and os.path.exists(local_last):
    # #133 Phase 2 の学習が開始済み → 中断再開
    pretrained_weights = local_last
    print(f'
[再開] Issue #133 Phase 2 の前回学習から再開: {pretrained_weights}')
    print(f'  サイズ: {os.path.getsize(pretrained_weights)/1024:.1f} KB')
elif os.path.exists(local_last) and not is_issue133_started:
    # _last.weightsがあるが#133 マーカーなし → 旧 Issue (#131 等) の重み → 無視
    print(f'
[スキップ] 旧 Issue (#131 等) の重みを検出しましたが、#133 Phase 2 初回実行のため無視します')
    print(f'  無視: {local_last}')
    print(f'  理由: .issue133_started マーカーが存在しないため')

if not pretrained_weights:
    # 新規開始: 事前学習済み重みを探索
    if os.path.exists(backbone_weights):
        pretrained_weights = backbone_weights
        print(f'
[新規] backbone重みで転移学習: {pretrained_weights}')
        print(f'  サイズ: {os.path.getsize(pretrained_weights)/1024:.1f} KB')
    elif os.path.exists(coco_weights):
        pretrained_weights = coco_weights
        print(f'
[新規] COCOフル重みで転移学習: {pretrained_weights}')
        print(f'  サイズ: {os.path.getsize(pretrained_weights)/1024:.1f} KB')
        print('  NOTE: darknetはcfg不一致の層を自動的にスキップして読み込みます')
    else:
        for wf in sorted(glob.glob(os.path.join(DARKNET_DIR, 'ModelZoo', '**', '*.weights'), recursive=True)):
            pretrained_weights = wf
            print(f'
[新規] ModelZoo内の重みで転移学習: {pretrained_weights}')
            print(f'  サイズ: {os.path.getsize(pretrained_weights)/1024:.1f} KB')
            break

if not pretrained_weights:
    print('
WARNING: 事前学習済み重みが見つかりません。スクラッチ学習を実行します。')


# --- Phase 2 (#133) 追加ロジック: #131 の best weights を転移学習元に使う ---
# .issue133_started が無い = Phase 2 初回実行時、
# もし #131 の best weights が Drive にあれば COCO backbone より優先する
issue131_best_drive = os.path.join(GDRIVE_BACKUP, 'yolo-fastest-person-192_best.weights')
issue131_best_local = os.path.join(DARKNET_DIR, 'yolo-fastest-person-192-131_best.weights')
if not is_issue133_started and os.path.exists(issue131_best_drive):
    # Drive からローカルにコピーして優先利用
    import shutil as _sh
    if not os.path.exists(issue131_best_local):
        _sh.copy2(issue131_best_drive, issue131_best_local)
        print(f'\n[Phase 2] #131 best weights を転移学習元として使用: {issue131_best_local}')
    pretrained_weights = issue131_best_local

# マーカーファイルを作成 (#133 Phase 2 学習開始を記録)
if not is_issue133_started:
    with open(ISSUE133_MARKER, 'w') as f:
        from datetime import datetime
        f.write(f'Issue #133 Phase 2 training started at {datetime.now().isoformat()}
')
        f.write(f'Initial weights: {pretrained_weights}
')
    print(f'
[マーカー] .issue133_started を作成しました（次回以降は中断再開モード）')

print(f'
--- 学習設定 ---')
print(f'  cfg: {CFG_PATH}')
print(f'  data: {DATA_PATH}')
print(f'  weights: {pretrained_weights if pretrained_weights else "(なし - スクラッチ)"}')
# 重みパスを環境変数としてbashセルに渡す
os.environ['TRAIN_WEIGHTS'] = pretrained_weights


In [ ]:
import subprocess
import sys
import re
import time
import os

DARKNET_DIR = '/content/Yolo-Fastest'
CFG = 'cfg/yolo-fastest-person-192.cfg'
DATA = 'data/person/obj.data'
WEIGHTS = os.environ.get('TRAIN_WEIGHTS', '')

# フォールバック: 直接ファイル検索
if not WEIGHTS:
    for candidate in [
        'backup/yolo-fastest-person-192_last.weights',
        'yolo-fastest-backbone.conv',
        'yolo-fastest-coco-pretrained.weights',
    ]:
        if os.path.exists(os.path.join(DARKNET_DIR, candidate)):
            WEIGHTS = candidate
            break

# 学習モード表示
if WEIGHTS:
    if '_last.weights' in WEIGHTS:
        print('=== 前回の学習から再開 ===')
    else:
        print('=== 転移学習 (事前学習済み重みから開始) ===')
else:
    print('=== スクラッチ学習 ===')

print(f'cfg:     {CFG}')
print(f'data:    {DATA}')
print(f'weights: {WEIGHTS or "(none)"}')
print()

# cfgからmax_batchesを取得 (進捗率計算用)
max_batches = 100000
cfg_path = os.path.join(DARKNET_DIR, CFG)
with open(cfg_path) as f:
    for line in f:
        m = re.match(r'max_batches\s*=\s*(\d+)', line.strip())
        if m:
            max_batches = int(m.group(1))
            break
print(f'--- 学習パラメータ (改善版 #131) ---')
print(f'  max_batches: {max_batches}')
print()
print(f'学習開始... ({max_batches:,} iterations)')
print()

# darknet学習コマンド
cmd = ['./darknet', 'detector', 'train', DATA, CFG]
if WEIGHTS:
    cmd.append(WEIGHTS)
cmd.extend(['-dont_show', '-gpus', '0'])

# リアルタイム出力 + 進捗表示
start_time = time.time()
last_progress_time = 0
log_file = open('/content/training_log.txt', 'w')

proc = subprocess.Popen(
    cmd,
    cwd=DARKNET_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
    universal_newlines=True
)

for line in proc.stdout:
    log_file.write(line)
    log_file.flush()

    # darknetの進捗行をパース (例: " 1000: 1.234, 1.567 avg loss, 0.001 rate, ...")
    m = re.match(r'\s*(\d+)[,:] ([\d.]+)[, ]+([\d.]+) avg', line)
    if m:
        iteration = int(m.group(1))
        loss = float(m.group(2))
        avg_loss = float(m.group(3))
        progress = iteration / max_batches * 100
        elapsed = time.time() - start_time
        elapsed_min = elapsed / 60

        # 推定残り時間
        if iteration > 0:
            eta_min = elapsed_min / progress * (100 - progress)
            eta_str = f'{eta_min:.0f}min' if eta_min < 60 else f'{eta_min/60:.1f}h'
        else:
            eta_str = '---'

        now = time.time()
        # 1000イテレーションごと、または5分ごとに進捗表示
        if iteration % 1000 == 0 or (now - last_progress_time) > 300:
            print(f'[{progress:5.1f}%] iter {iteration:>6,}/{max_batches:,} | '
                  f'loss: {loss:.3f} | avg_loss: {avg_loss:.3f} | '
                  f'elapsed: {elapsed_min:.0f}min | ETA: {eta_str}')
            sys.stdout.flush()
            last_progress_time = now
    elif 'Saving weights' in line or 'avg loss' in line.lower():
        # 重み保存時も表示
        print(line.rstrip())
        sys.stdout.flush()

proc.wait()
log_file.close()

elapsed_total = (time.time() - start_time) / 60
print(f'
=== 学習完了 (所要時間: {elapsed_total:.1f}分) ===')
print(f'ログ: /content/training_log.txt')
if proc.returncode != 0:
    print(f'WARNING: darknet 終了コード {proc.returncode}')


In [ ]:
# 学習結果をGoogle Driveにバックアップ
import shutil
import os
import glob

BACKUP_DIR = '/content/Yolo-Fastest/backup'
GDRIVE_BACKUP = '/content/drive/MyDrive/yolo_fastest_darknet_person'

print('=== Google Driveへバックアップ ===')

# 重みファイルをコピー
for wf in glob.glob(os.path.join(BACKUP_DIR, '*.weights')):
    dst = os.path.join(GDRIVE_BACKUP, os.path.basename(wf))
    shutil.copy2(wf, dst)
    size_kb = os.path.getsize(wf) / 1024
    print(f'  {os.path.basename(wf)}: {size_kb:.1f} KB')

# 学習ログもコピー
log_file = '/content/training_log.txt'
if os.path.exists(log_file):
    shutil.copy2(log_file, os.path.join(GDRIVE_BACKUP, 'training_log.txt'))
    print(f'  training_log.txt')

# cfgもコピー
cfg_path = '/content/Yolo-Fastest/cfg/yolo-fastest-person-192.cfg'
if os.path.exists(cfg_path):
    shutil.copy2(cfg_path, os.path.join(GDRIVE_BACKUP, 'yolo-fastest-person-192.cfg'))
    print(f'  yolo-fastest-person-192.cfg')

# chart.png (darknetが生成する学習曲線)
chart_png = '/content/Yolo-Fastest/chart.png'
if os.path.exists(chart_png):
    shutil.copy2(chart_png, os.path.join(GDRIVE_BACKUP, 'chart.png'))
    print(f'  chart.png')

print(f'\n保存先: {GDRIVE_BACKUP}')

---
## Step 7: 精度評価

In [ ]:
%%bash
cd /content/Yolo-Fastest

CFG="cfg/yolo-fastest-person-192.cfg"
DATA="data/person/obj.data"

# best weightsを使用
BEST="backup/yolo-fastest-person-192_best.weights"
if [ ! -f "$BEST" ]; then
    # Google Driveから復元
    GDRIVE_BEST="/content/drive/MyDrive/yolo_fastest_darknet_person/yolo-fastest-person-192_best.weights"
    if [ -f "$GDRIVE_BEST" ]; then
        cp "$GDRIVE_BEST" "$BEST"
        echo "Google Driveからbest weightsを復元しました"
    else
        # final weightsを試す
        BEST="backup/yolo-fastest-person-192_final.weights"
    fi
fi

if [ -f "$BEST" ]; then
    echo "=== 精度評価: $BEST ==="
    echo ""

    # 評価用にcfgをテストモードに変更
    cp $CFG cfg/yolo-fastest-person-192-test.cfg
    sed -i 's/batch=64/batch=1/' cfg/yolo-fastest-person-192-test.cfg
    sed -i 's/subdivisions=16/subdivisions=1/' cfg/yolo-fastest-person-192-test.cfg

    ./darknet detector map $DATA cfg/yolo-fastest-person-192-test.cfg $BEST
else
    echo "ERROR: best weightsが見つかりません"
    echo "学習 (Step 6) を先に実行してください"
fi

---
## Step 8: darknet weights -> TFLite INT8 変換

これが最も重要なステップです。

### 変換パス
```
darknet .weights -> Keras .h5 -> TFLite FP32 -> TFLite INT8
```

### 使用ツール
- [david8862/keras-YOLOv3-model-set](https://github.com/david8862/keras-YOLOv3-model-set)

このツールはYOLOv3系のcfg/weightsをKerasに変換する機能を提供しています。

In [ ]:
# keras-YOLOv3-model-set のセットアップ
import os

CONVERTER_DIR = '/content/keras-YOLOv3-model-set'

if not os.path.isdir(CONVERTER_DIR):
    !git clone https://github.com/david8862/keras-YOLOv3-model-set.git {CONVERTER_DIR}
else:
    print(f'{CONVERTER_DIR} は既に存在します')

# 依存パッケージ
!pip install -q tensorflow keras matplotlib pillow tf-keras

# NumPy 2.0互換性修正: np.product -> np.prod
import subprocess
result = subprocess.run(['grep', '-rl', 'np.product', CONVERTER_DIR], capture_output=True, text=True)
if result.stdout.strip():
    !grep -rl 'np.product' {CONVERTER_DIR} | xargs sed -i 's/np.product/np.prod/g'
    print('np.product -> np.prod 修正済み')

# Keras 3.x互換性修正: tf-keras (Keras 2) を使用
convert_py = os.path.join(CONVERTER_DIR, 'tools', 'model_converter', 'convert.py')
with open(convert_py, 'r') as f:
    content = f.read()
content = content.replace('from tensorflow.keras', 'from tf_keras')
content = content.replace('from keras.', 'from tf_keras.')
content = content.replace('import keras', 'import tf_keras as keras')
with open(convert_py, 'w') as f:
    f.write(content)
print('convert.py を tf-keras 対応に修正済み')

print('=== セットアップ完了 ===')


In [ ]:
import os

DARKNET_DIR = '/content/Yolo-Fastest'
CONVERTER_DIR = '/content/keras-YOLOv3-model-set'
CFG_PATH = os.path.join(DARKNET_DIR, 'cfg', 'yolo-fastest-person-192.cfg')
BEST_WEIGHTS = os.path.join(DARKNET_DIR, 'backup', 'yolo-fastest-person-192_final.weights')
KERAS_H5_PATH = '/content/yolo_fastest_person.h5'

# Google Driveから復元 (必要に応じて)
if not os.path.exists(BEST_WEIGHTS):
    gdrive_best = '/content/drive/MyDrive/yolo_fastest_darknet_person/yolo-fastest-person-192_final.weights'
    if os.path.exists(gdrive_best):
        import shutil
        os.makedirs(os.path.dirname(BEST_WEIGHTS), exist_ok=True)
        shutil.copy2(gdrive_best, BEST_WEIGHTS)
        print(f'Google Driveから復元: {BEST_WEIGHTS}')

if not os.path.exists(BEST_WEIGHTS):
    print(f'ERROR: {BEST_WEIGHTS} が見つかりません')
    print('学習 (Step 6) を先に実行してください')
else:
    print(f'cfg: {CFG_PATH}')
    print(f'weights: {BEST_WEIGHTS} ({os.path.getsize(BEST_WEIGHTS)/1024:.1f} KB)')

    # Step 1: darknet -> Keras .h5 変換
    print('\n=== Step 8-1: darknet -> Keras .h5 ===')
    os.chdir(CONVERTER_DIR)
    !python tools/model_converter/convert.py \
        {CFG_PATH} \
        {BEST_WEIGHTS} \
        {KERAS_H5_PATH}

    if os.path.exists(KERAS_H5_PATH):
        print(f'\nKerasモデル保存: {KERAS_H5_PATH} ({os.path.getsize(KERAS_H5_PATH)/1024:.1f} KB)')
    else:
        print('ERROR: Kerasモデルの変換に失敗しました')
        print('\ncfgファイルが変換ツールと互換性がない可能性があります。')
        print('代替方法: 以下のセルで別の変換ツールを試します。')


In [ ]:
# 代替変換方法: Lebhoryi/yolo-fastest_inference を使用
# david8862で失敗した場合のみ実行
import os

KERAS_H5_PATH = '/content/yolo_fastest_person.h5'

if not os.path.exists(KERAS_H5_PATH):
    print('=== 代替変換: Lebhoryi/yolo-fastest_inference ===')

    ALT_DIR = '/content/yolo-fastest_inference'
    if not os.path.isdir(ALT_DIR):
        !git clone https://github.com/Lebhoryi/yolo-fastest_inference.git {ALT_DIR}

    os.chdir(ALT_DIR)
    !pip install -q configparser

    CFG_PATH = '/content/Yolo-Fastest/cfg/yolo-fastest-person-192.cfg'
    BEST_WEIGHTS = '/content/Yolo-Fastest/backup/yolo-fastest-person-192_best.weights'

    # このツールの変換スクリプトを確認
    !ls -la *.py convert/ 2>/dev/null || echo 'scriptsを確認中...'
    !find . -name '*.py' -type f | head -20

    print('\n上記のスクリプトを確認し、適切な変換コマンドを実行してください。')
else:
    print(f'Kerasモデルが既に存在します: {KERAS_H5_PATH}')
    print('このセルはスキップします。')

In [ ]:
import os
import numpy as np
import glob
from PIL import Image
import tf_keras
import tensorflow as tf

KERAS_H5_PATH = '/content/yolo_fastest_person.h5'
FP32_PATH = '/content/yolo_fastest_person_fp32.tflite'
INT8_PATH = '/content/yolo_fastest_person_darknet_int8.tflite'
DATASET_DIR = '/content/dataset'
IMG_SIZE = 192

if not os.path.exists(KERAS_H5_PATH):
    print('ERROR: Kerasモデルが見つかりません。Step 8-1を実行してください。')
else:
    # tf-keras (Keras 2) でモデルを読み込み
    print('=== Step 8-2: Keras -> TFLite FP32 ===')
    model = tf_keras.models.load_model(KERAS_H5_PATH, compile=False)

    # 動的入力形状を192x192x3に固定 (TFLite変換に必要)
    fixed_input = tf_keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='image_input')
    outputs = model(fixed_input)
    fixed_model = tf_keras.Model(inputs=fixed_input, outputs=outputs)
    fixed_model.summary(line_length=120)

    # FP32 TFLite
    converter = tf.lite.TFLiteConverter.from_keras_model(fixed_model)
    tflite_fp32 = converter.convert()
    with open(FP32_PATH, 'wb') as f:
        f.write(tflite_fp32)
    print(f'\nFP32 TFLite: {os.path.getsize(FP32_PATH)/1024:.1f} KB')

    # INT8量子化
    print('\n=== Step 8-3: TFLite INT8 量子化 ===')
    cal_dir = os.path.join(DATASET_DIR, 'images', 'val')
    cal_images = sorted(glob.glob(os.path.join(cal_dir, '*.jpg')) +
                        glob.glob(os.path.join(cal_dir, '*.png')))[:200]
    print(f'キャリブレーション画像: {len(cal_images)}枚')

    def representative_dataset():
        for img_path in cal_images:
            img = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
            arr = np.array(img, dtype=np.float32) / 255.0
            arr = arr.reshape(1, IMG_SIZE, IMG_SIZE, 3)
            yield [arr]

    converter = tf.lite.TFLiteConverter.from_keras_model(fixed_model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    try:
        int8_model = converter.convert()
        with open(INT8_PATH, 'wb') as f:
            f.write(int8_model)
        int8_kb = os.path.getsize(INT8_PATH) / 1024
        print(f'\nINT8 TFLite: {int8_kb:.1f} KB')
    except Exception as e:
        print(f'INT8 量子化エラー: {e}')

    # 入出力テンソル確認
    print('\n--- 入出力テンソル確認 ---')
    interp = tf.lite.Interpreter(model_path=INT8_PATH)
    interp.allocate_tensors()
    print('入力:')
    for d in interp.get_input_details():
        qp = d.get('quantization_parameters', {})
        print(f'  {d["name"]}: shape={d["shape"]}, scale={qp["scales"][0]:.8f}, zp={qp["zero_points"][0]}')
    print('出力:')
    for d in interp.get_output_details():
        qp = d.get('quantization_parameters', {})
        print(f'  {d["name"]}: shape={d["shape"]}, scale={qp["scales"][0]:.8f}, zp={qp["zero_points"][0]}')


---
## Step 9: モデル詳細確認

INT8 TFLiteモデルの入出力詳細を確認します。

**重要**: 出力テンソルのscale/zero_pointはMCU側の後処理コードに必要です。必ず記録してください。

In [ ]:
import tensorflow as tf
import numpy as np
import os

INT8_PATH = '/content/yolo_fastest_person_darknet_int8.tflite'

if not os.path.exists(INT8_PATH):
    print(f'ERROR: {INT8_PATH} が見つかりません。Step 8を実行してください。')
else:
    file_size_kb = os.path.getsize(INT8_PATH) / 1024
    print('=' * 60)
    print('INT8 TFLite モデル詳細')
    print('=' * 60)
    print(f'ファイル: {INT8_PATH}')
    print(f'サイズ: {file_size_kb:.1f} KB ({file_size_kb/1024:.2f} MB)')
    print(f'NPUアリーナ制約 (432KB): {"OK" if file_size_kb <= 432 else "OVER"}')
    print()

    interp = tf.lite.Interpreter(model_path=INT8_PATH)
    interp.allocate_tensors()

    # 入力詳細
    print('--- 入力テンソル ---')
    for i, d in enumerate(interp.get_input_details()):
        print(f'  [{i}] name: {d["name"]}')
        print(f'       shape: {d["shape"]}')
        print(f'       dtype: {d["dtype"]}')
        qp = d.get('quantization_parameters', {})
        sc = qp.get('scales', np.array([]))
        zp = qp.get('zero_points', np.array([]))
        if len(sc) > 0:
            print(f'       scale: {sc[0]:.8f}')
            print(f'       zero_point: {zp[0]}')
    print()

    # 出力詳細
    print('--- 出力テンソル ---')
    for i, d in enumerate(interp.get_output_details()):
        print(f'  [{i}] name: {d["name"]}')
        print(f'       shape: {d["shape"]}')
        print(f'       dtype: {d["dtype"]}')
        qp = d.get('quantization_parameters', {})
        sc = qp.get('scales', np.array([]))
        zp = qp.get('zero_points', np.array([]))
        if len(sc) > 0:
            print(f'       scale: {sc[0]:.8f}')
            print(f'       zero_point: {zp[0]}')
    print()

    # オペレータ一覧
    print('--- オペレータ一覧 ---')
    ops = set()
    tensor_details = interp.get_tensor_details()
    print(f'  テンソル数: {len(tensor_details)}')
    # TFLiteのオペレータ名は直接取得できないが、テンソル名から推定
    for td in tensor_details:
        name = td['name']
        for op_name in ['Conv2D', 'DepthwiseConv2D', 'MaxPool', 'Concatenate',
                        'Upsample', 'Reshape', 'LeakyRelu', 'Add', 'Mul',
                        'Sigmoid', 'StridedSlice', 'Pad', 'Resize']:
            if op_name.lower() in name.lower():
                ops.add(op_name)
    if ops:
        print(f'  検出されたオペレータ: {", ".join(sorted(ops))}')

    # Ethos-U55未対応オペレータの確認
    unsupported = ops & {'StridedSlice', 'Resize'}
    if unsupported:
        print(f'\n  WARNING: Ethos-U55未対応の可能性があるオペレータ: {unsupported}')
    else:
        print(f'\n  OK: Ethos-U55未対応オペレータは検出されませんでした')

    print()
    print('=' * 60)
    print('*** 重要: 上記の出力テンソルの scale / zero_point を控えてください ***')
    print('MCU側の後処理コード (DetectorPostProcessing.cc) に設定が必要です')
    print('=' * 60)

---
## Step 9.5: MCUパラメータ出力 (改善版 #131 追加)

再学習後のモデルから、MCU側コードに設定すべきパラメータを出力します。

### 更新が必要なファイル
1. `fall_detection_postprocess.h` - 量子化パラメータ (scale, zero_point)
2. `fall_detection_postprocess.c` - アンカー値 (anchors)

### 手順
1. 以下のセルを実行してパラメータを確認
2. MCU側コードの該当箇所を手動で更新

In [ ]:
# === MCU側コード更新用パラメータ出力 (改善版 #131 追加) ===
# 再学習後のモデルから量子化パラメータとアンカー値を出力
import os
import re

print('=' * 70)
print('MCU側コード更新用パラメータ')
print('=' * 70)

# --- 1. 量子化パラメータ (TFLite INT8モデルから) ---
INT8_PATH = '/content/yolo_fastest_person_darknet_int8.tflite'
if os.path.exists(INT8_PATH):
    import tensorflow as tf
    import numpy as np

    interp = tf.lite.Interpreter(model_path=INT8_PATH)
    interp.allocate_tensors()

    print('\n--- 量子化パラメータ ---')
    print('更新先: fall_detection_postprocess.h')
    print()

    outputs = interp.get_output_details()
    for i, d in enumerate(outputs):
        qp = d.get('quantization_parameters', {})
        sc = qp.get('scales', np.array([]))
        zp = qp.get('zero_points', np.array([]))
        shape = d['shape']
        name = d['name']
        if len(sc) > 0:
            # Branch判定: shapeから grid サイズを推定
            # Branch 0: 6x6 (大きいstride), Branch 1: 12x12 (小さいstride)
            grid_size = shape[1] if len(shape) >= 3 else 0
            branch = 0 if grid_size <= 6 else 1
            print(f'  Output [{i}] "{name}": shape={shape}')
            print(f'    -> Branch {branch} ({grid_size}x{grid_size})')
            print(f'    #define POSTPROC_BRANCH{branch}_SCALE       ({sc[0]:.8f}f)')
            print(f'    #define POSTPROC_BRANCH{branch}_ZERO_POINT  ({zp[0]})')
            print()
else:
    print('\nWARNING: INT8モデルが見つかりません。Step 8を先に実行してください。')

# --- 2. アンカー値 (cfgファイルから) ---
CFG_PATH = '/content/Yolo-Fastest/cfg/yolo-fastest-person-192.cfg'
if os.path.exists(CFG_PATH):
    print('\n--- アンカー値 ---')
    print('更新先: fall_detection_postprocess.c')
    print()

    with open(CFG_PATH) as f:
        cfg_lines = f.readlines()

    # cfgからanchorsとmaskを抽出
    anchors_str = None
    masks = []
    for line in cfg_lines:
        line = line.strip()
        if line.startswith('anchors=') and anchors_str is None:
            anchors_str = line.split('=')[1].strip()
        if line.startswith('mask='):
            masks.append(line.split('=')[1].strip())

    if anchors_str:
        # アンカーをパース
        anchor_vals = [float(x.strip()) for x in anchors_str.split(',')]
        anchors = [(anchor_vals[i], anchor_vals[i+1]) for i in range(0, len(anchor_vals), 2)]

        print(f'  全アンカー (cfgから): {anchors_str}')
        print(f'  パース結果: {anchors}')
        print()

        # Branch 0 (mask=3,4,5) と Branch 1 (mask=0,1,2) に分割
        for branch_idx, mask_str in enumerate(masks):
            mask_indices = [int(x.strip()) for x in mask_str.split(',')]
            branch_anchors = [anchors[m] for m in mask_indices]
            grid = '6x6' if branch_idx == 0 else '12x12'
            print(f'  Branch {branch_idx} ({grid}, mask={mask_str}):')
            for a_idx, (w, h) in enumerate(branch_anchors):
                print(f'    s_config.branches[{branch_idx}].anchors[{a_idx}][0] = {w:.1f}f;  '
                      f's_config.branches[{branch_idx}].anchors[{a_idx}][1] = {h:.1f}f;')
            print()
else:
    print('\nWARNING: cfgファイルが見つかりません。Step 3を先に実行してください。')

print('=' * 70)
print('上記の値をMCU側コードにコピーしてください。')
print('  fall_detection_postprocess.h: 量子化パラメータ (#define)')
print('  fall_detection_postprocess.c: アンカー値 (s_config.branches)')
print('=' * 70)

---
## Step 10: Vela互換性確認

Arm VelaコンパイラでEthos-U55との互換性を確認します。

確認ポイント:
- NPUに配置されるオペレータの割合 (100%が理想)
- CPUフォールバックオペレータの有無
- サブグラフ数 (1が理想)

In [ ]:
# Arm Vela コンパイラのインストールと確認
!pip install -q ethos-u-vela

print('=== Velaバージョン ===')
!vela --version 2>/dev/null || echo 'Velaのインストールに失敗しました'

In [ ]:
%%bash
INT8_PATH="/content/yolo_fastest_person_darknet_int8.tflite"
VELA_OUTPUT="/content/vela_output"

if [ ! -f "$INT8_PATH" ]; then
    echo "ERROR: INT8モデルが見つかりません。Step 8を実行してください。"
    exit 0
fi

mkdir -p $VELA_OUTPUT

echo "=== Velaコンパイル (Ethos-U55-256) ==="
echo ""

vela \
    --accelerator-config ethos-u55-256 \
    --system-config Ethos_U55_High_End_Embedded \
    --memory-mode Sram_Only \
    --output-dir $VELA_OUTPUT \
    $INT8_PATH 2>&1

echo ""
echo "=== Vela出力ファイル ==="
ls -la $VELA_OUTPUT/ 2>/dev/null

# Vela最適化済みモデルのサイズ確認
VELA_MODEL=$(find $VELA_OUTPUT -name '*.tflite' | head -1)
if [ -n "$VELA_MODEL" ]; then
    SIZE_KB=$(du -k "$VELA_MODEL" | cut -f1)
    echo ""
    echo "Vela最適化後サイズ: ${SIZE_KB} KB"
    if [ $SIZE_KB -le 432 ]; then
        echo "OK: NPUアリーナ制約 (432KB) 以内"
    else
        echo "WARNING: NPUアリーナ制約 (432KB) 超過"
    fi
fi

---
## Step 11: 成果物ダウンロード

In [ ]:
# Google Drive に全成果物を保存
import shutil
import os
import json

OUTPUT_DIR = '/content/drive/MyDrive/yolo_fastest_darknet_person/model'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('=== 成果物をGoogle Driveに保存 ===')

# コピー対象ファイル
files_to_copy = {
    '/content/Yolo-Fastest/backup/yolo-fastest-person-192_best.weights': 'yolo-fastest-person-192_best.weights',
    '/content/yolo_fastest_person.h5': 'yolo_fastest_person.h5',
    '/content/yolo_fastest_person_fp32.tflite': 'yolo_fastest_person_fp32.tflite',
    '/content/yolo_fastest_person_darknet_int8.tflite': 'yolo_fastest_person_darknet_int8.tflite',
    '/content/Yolo-Fastest/cfg/yolo-fastest-person-192.cfg': 'yolo-fastest-person-192.cfg',
    '/content/Yolo-Fastest/chart.png': 'training_chart.png',
    '/content/training_log.txt': 'training_log.txt',
}

# Vela出力
import glob
vela_models = glob.glob('/content/vela_output/*.tflite')
for vm in vela_models:
    files_to_copy[vm] = f'vela_{os.path.basename(vm)}'

for src, dst_name in files_to_copy.items():
    if os.path.exists(src):
        dst = os.path.join(OUTPUT_DIR, dst_name)
        shutil.copy2(src, dst)
        size_kb = os.path.getsize(src) / 1024
        print(f'  {dst_name}: {size_kb:.1f} KB')
    else:
        print(f'  SKIP: {dst_name} (ファイルなし)')

# 量子化パラメータをJSONで保存
INT8_PATH = '/content/yolo_fastest_person_darknet_int8.tflite'
if os.path.exists(INT8_PATH):
    import tensorflow as tf
    import numpy as np

    interp = tf.lite.Interpreter(model_path=INT8_PATH)
    interp.allocate_tensors()

    quant_info = {'input': [], 'output': []}
    for tag, details in [('input', interp.get_input_details()),
                         ('output', interp.get_output_details())]:
        for d in details:
            qp = d.get('quantization_parameters', {})
            sc = qp.get('scales', np.array([]))
            zp = qp.get('zero_points', np.array([]))
            info = {
                'name': d['name'],
                'shape': d['shape'].tolist(),
                'dtype': str(d['dtype']),
                'scale': float(sc[0]) if len(sc) > 0 else None,
                'zero_point': int(zp[0]) if len(zp) > 0 else None,
            }
            quant_info[tag].append(info)

    quant_json_path = os.path.join(OUTPUT_DIR, 'quantization_params.json')
    with open(quant_json_path, 'w') as f:
        json.dump(quant_info, f, indent=2)
    print(f'  quantization_params.json')

print(f'\n=== 保存先: {OUTPUT_DIR} ===')

In [ ]:
# INT8 TFLite モデルをダウンロード
from google.colab import files
import os

INT8_PATH = '/content/yolo_fastest_person_darknet_int8.tflite'

if os.path.exists(INT8_PATH):
    files.download(INT8_PATH)
    print('INT8モデルのダウンロードを開始しました')
else:
    print('INT8モデルが見つかりません。Step 8を先に実行してください。')

---
## まとめ

### 改善版 (#131) の変更サマリ

| 項目 | 旧版 (#128) | 改善版 (#131) |
|---|---|---|
| max_batches | 10,000 | **100,000** |
| burn_in (warmup) | 1,000 | **2,000** |
| steps (LR decay) | 8000,9000 | **60000,80000,90000** |
| データ拡張 angle | 0 | **15** |
| ignore_thresh | 0.7 | **0.5** |
| 事前学習重み | なし | **COCO backbone転移学習** |
| MCUパラメータ出力 | なし | **自動出力セル追加** |

### 変換パス
```
darknet .weights  -->  Keras .h5  -->  TFLite FP32  -->  TFLite INT8
   (dog-qiuqiu)    (david8862)       (TF Lite)        (PTQ, INT8)
```

### 次のステップ (実機デプロイ)

1. INT8 TFLiteモデルをダウンロード
2. RUHMI `mcu_deploy.py --ethos` でMERA変換
3. **単一サブグラフ (sub_0000のみ) であることを確認**
4. MCU側パラメータ更新 (Step 9.5の出力値を使用):
   - `fall_detection_postprocess.h`: 量子化パラメータ (scale, zero_point)
   - `fall_detection_postprocess.c`: アンカー値 (anchors)
5. e2studioビルド -> 実機書き込み -> 動作確認

### 後処理パラメータ (MCUコードに設定が必要)
- 入力: 192x192x3 RGB (INT8)
- 出力: 2ブランチ (6x6 stride-32 + 12x12 stride-16)
- アンカー: Step 4で計算した値 + Step 9.5で出力
- 出力テンソルのscale/zero_point: Step 9.5で出力
- デコード: YOLOv3スタイル
  - objectness = sigmoid((int8_val - zero_point) * scale)
  - bbox: sigmoid(tx)+cx, sigmoid(ty)+cy, exp(tw)*anchor_w, exp(th)*anchor_h
  - class_score = sigmoid(class_val) * objectness
  - NMS (IoU閾値 0.45, 信頼度閾値 0.5)

### トラブルシューティング

| 問題 | 対処 |
|---|---|
| darknetコンパイル失敗 | OpenCVバージョン確認、`OPENCV=0`で再コンパイル |
| Keras変換失敗 | cfg形式の互換性確認、Lebhoryi/yolo-fastest_inferenceを代替 |
| INT8量子化エラー | FP32 TFLiteのオペレータ確認、キャリブレーション画像数を増やす |
| Velaコンパイル失敗 | 非対応オペレータの特定、cfgの活性化関数確認 (LeakyReLU/ReLU6のみ使用) |
| モデルサイズ>432KB | フィルタ数削減、またはSDRAM配置を検討 |
| 精度が改善しない | 事前学習済み重みの確認、max_batches をさらに増加 (200,000) |